# Módulo 2 — Desarrollador Full Stack de Soluciones Inteligentes

## Unidad I: Redes Neuronales y Aplicaciones Full Stack

### Clase 3: Creacion de API del proyecto (FastAPI) para que el frontend pueda consumir predicciones

---



### Actualización del archivo `requirements.txt` con las dependencias de la API

tensorflow
pandas
numpy
scikit-learn
joblib
matplotlib
openml
fastapi
uvicorn[standard]

### Creación del modulo `src/api.py`

In [ ]:
# API de predicción de ingresos (>50K / <=50K) - Versión corta para clase de 1h

# Librería estándar para interactuar con el sistema operativo (archivos, rutas)
import os

# Permite cargar objetos serializados (preprocessor entrenado)
import joblib

# Librería numérica base
import numpy as np

# Manejo de datos en formato tabla (DataFrame)
import pandas as pd

# Framework donde se entrenó el modelo
import tensorflow as tf

# Permite ejecutar código al iniciar y cerrar la aplicación
from contextlib import asynccontextmanager

# Framework para construir APIs modernas en Python
from fastapi import FastAPI, HTTPException

# Middleware para permitir conexiones desde frontend (CORS)
from fastapi.middleware.cors import CORSMiddleware

# Define el formato de datos que recibirá la API (validación automática)
from pydantic import BaseModel

# Columnas definidas en el proyecto durante entrenamiento
from src.config import NUMERIC_FEATURES, CATEGORICAL_FEATURES


# Rutas donde se guardaron los artefactos al entrenar
PREPROCESSOR_PATH = "artifacts/preprocessor.joblib"
MODEL_PATH = "artifacts/model.keras"

# Variables globales donde se almacenarán modelo y preprocesador cargados
preprocessor = None
model = None


# Función que carga los artefactos entrenados
def load_artifacts():
    global preprocessor, model  # permite modificar variables globales

    # Verifica que los archivos existan
    if not os.path.isfile(PREPROCESSOR_PATH) or not os.path.isfile(MODEL_PATH):
        raise FileNotFoundError("Falta entrenar: python -m src.train")

    # Carga el preprocesador entrenado
    preprocessor = joblib.load(PREPROCESSOR_PATH)

    # Carga el modelo entrenado
    model = tf.keras.models.load_model(MODEL_PATH)


# Función que se ejecuta al iniciar y cerrar la API
@asynccontextmanager
async def lifespan(app: FastAPI):

    # Startup: cargar modelo antes de aceptar peticiones
    try:
        load_artifacts()
    except FileNotFoundError as e:
        print("Aviso:", e)

    yield  # Aquí la API queda corriendo

    # Shutdown: aquí se podrían liberar recursos si fuera necesario


# Crear aplicación FastAPI
app = FastAPI(title="Income ML API", version="1.0", lifespan=lifespan)

# Permitir que el frontend local pueda hacer peticiones a esta API
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000", "http://localhost:5173"],
    allow_methods=["*"],
    allow_headers=["*"]
)


# Define el formato JSON que debe enviar el frontend
# FastAPI validará automáticamente los tipos
class PredictRequest(BaseModel):
    age: int
    fnlwgt: int
    education_num: int
    capital_gain: int
    capital_loss: int
    hours_per_week: int
    workclass: str
    education: str
    marital_status: str
    occupation: str
    relationship: str
    race: str
    sex: str
    native_country: str


# Endpoint raíz: verificar estado de la API
@app.get("/")
def root():
    """Página principal: estado del modelo."""
    return {
        "message": "Income ML API",
        "docs": "/docs",
        "model_loaded": model is not None
    }


# Endpoint que devuelve las variables del modelo
@app.get("/features")
def get_features():
    """Lista de variables y categorías disponibles."""

    # Si aún no se cargó el modelo, intenta cargarlo
    if preprocessor is None:
        try:
            load_artifacts()
        except FileNotFoundError:
            raise HTTPException(503, "Entrena antes: python -m src.train")

    opts = {}

    # Accede al transformador categórico dentro del preprocessor
    cat = preprocessor.named_transformers_.get("cat")

    # Extrae las categorías aprendidas por el OneHotEncoder
    if cat and hasattr(cat.named_steps.get("encoder"), "categories_"):
        for name, arr in zip(CATEGORICAL_FEATURES, cat.named_steps["encoder"].categories_):
            opts[name] = [str(x) for x in arr]

    # Devuelve lista de variables para que el frontend construya formulario dinámico
    return {
        "numeric": NUMERIC_FEATURES,
        "categorical": list(CATEGORICAL_FEATURES),
        "categorical_options": opts
    }


# Endpoint principal de predicción
@app.post("/predict")
def predict(body: PredictRequest):
    """Recibe datos de una persona y devuelve predicción."""

    # Verifica que modelo y preprocesador estén cargados
    if preprocessor is None or model is None:
        try:
            load_artifacts()
        except FileNotFoundError:
            raise HTTPException(503, "Modelo no cargado. Ejecuta: python -m src.train")

    # Construye un diccionario con los datos recibidos
    row = {
        "age": body.age,
        "fnlwgt": body.fnlwgt,
        "education-num": body.education_num,
        "capital-gain": body.capital_gain,
        "capital-loss": body.capital_loss,
        "hours-per-week": body.hours_per_week,
        "workclass": body.workclass,
        "education": body.education,
        "marital-status": body.marital_status,
        "occupation": body.occupation,
        "relationship": body.relationship,
        "race": body.race,
        "sex": body.sex,
        "native-country": body.native_country,
    }

    # Convertir a DataFrame porque el preprocessor espera formato tabular
    X = preprocessor.transform(pd.DataFrame([row]))

    # Convertir a matriz densa si es dispersa
    if hasattr(X, "toarray"):
        X = X.toarray()

    # TensorFlow requiere float32
    X = np.asarray(X, dtype=np.float32)

    # Obtener probabilidad (sigmoid devuelve valor entre 0 y 1)
    prob = float(model.predict(X, verbose=0)[0][0])

    # Devolver respuesta en formato JSON
    return {
        "prediction": ">50K" if prob > 0.5 else "<=50K",
        "probability": round(prob, 4)
    }


# Permite ejecutar la API directamente con: python src/api.py
if __name__ == "__main__":
    import uvicorn
    uvicorn.run("src.api:app", host="127.0.0.1", port=8000, reload=True)

## Creación del Front con React

1. Instalar Node.js versión LTS

    `https://nodejs.org/en`

2. Verificar la instalación

    `node -v`

    `npm -v`

3. En la carpeta `INCOME-ML-SYSTEM ejecutar

    `npm create vite@latest income-ml-frontend -- --template react`

4. Cambiar la carpeta de cache de npm desde PowerShell

    `npm config set cache "C:\Users\luisg\npm-cache"`

5. Instala las dependencias del frontend

    `npm install`

6. Instalar dependencias del proyecto

    `npm install react react-dom`
    `npm install -D @vitejs/plugin-react @types/react @types/react-dom`

5. Cambiar directorio

    `cd income-ml-frontend`

6. Levantar el Front

    `npm run dev`

## Creacion de componentes

`income-ml-frontend/app/lib/api.ts`

In [ ]:
/**
 * URL base de la API de predicción (backend FastAPI).
 * En desarrollo: el frontend corre en localhost:5173 y la API en localhost:8000.
 * Todas las llamadas HTTP del frontend usarán esta base.
 */
 
export const API_BASE = "http://localhost:8000";


/**
 * Estructura del JSON que el frontend envía al backend
 * cuando hace POST a /predict.
 * 
 * Debe coincidir EXACTAMENTE con lo que espera FastAPI.
 */
 
export interface PredictRequest {
  age: number;               // Edad
  fnlwgt: number;            // Peso final del censo (variable técnica del dataset)
  education_num: number;     // Nivel educativo en formato numérico
  capital_gain: number;      // Ganancia de capital
  capital_loss: number;      // Pérdida de capital
  hours_per_week: number;    // Horas trabajadas por semana
  workclass: string;         // Tipo de empleo
  education: string;         // Nivel educativo (texto)
  marital_status: string;    // Estado civil
  occupation: string;        // Ocupación
  relationship: string;      // Relación familiar
  race: string;              // Raza
  sex: string;               // Sexo
  native_country: string;    // País de origen
}


/**
 * Estructura de la respuesta que devuelve el backend
 * después de hacer la predicción.
 */
 
export interface PredictResponse {
  prediction: ">50K" | "<=50K"; // Clase predicha (mayor o menor a 50K)
  probability: number;          // Probabilidad estimada por el modelo (0–1)
}


/**
 * Estructura de la respuesta del endpoint GET /features.
 * 
 * Sirve para que el frontend construya dinámicamente el formulario.
 */
 
export interface FeaturesResponse {
  numeric: string[];   // Lista de variables numéricas del modelo
  categorical: string[];  // Lista de variables categóricas
  categorical_options: Record<string, string[]>;   // Opciones posibles para cada variable categórica
}

`income-ml-frontend/app/lib/options.ts`

In [ ]:
/**
 * Opciones por defecto para variables categóricas del dataset Adult (OpenML 1590).
 * Se usan cuando la API aún no tiene el modelo entrenado y no devuelve /features.
 */
export const DEFAULT_CATEGORICAL_OPTIONS: Record<string, string[]> = {
  workclass: [
    "Private",
    "Self-emp-not-inc",
    "Self-emp-inc",
    "Federal-gov",
    "Local-gov",
    "State-gov",
    "Without-pay",
    "Never-worked",
  ],
  education: [
    "Bachelors",
    "Some-college",
    "11th",
    "HS-grad",
    "Prof-school",
    "Assoc-acdm",
    "Assoc-voc",
    "9th",
    "7th-8th",
    "12th",
    "Masters",
    "1st-4th",
    "10th",
    "Doctorate",
    "5th-6th",
    "Preschool",
  ],
  "marital-status": [
    "Married-civ-spouse",
    "Divorced",
    "Never-married",
    "Separated",
    "Widowed",
    "Married-spouse-absent",
    "Married-AF-spouse",
  ],
  occupation: [
    "Tech-support",
    "Craft-repair",
    "Other-service",
    "Sales",
    "Exec-managerial",
    "Prof-specialty",
    "Handlers-cleaners",
    "Machine-op-inspct",
    "Adm-clerical",
    "Farming-fishing",
    "Transport-moving",
    "Priv-house-serv",
    "Protective-serv",
    "Armed-Forces",
  ],
  relationship: [
    "Wife",
    "Own-child",
    "Husband",
    "Not-in-family",
    "Other-relative",
    "Unmarried",
  ],
  race: [
    "White",
    "Asian-Pac-Islander",
    "Amer-Indian-Eskimo",
    "Other",
    "Black",
  ],
  sex: ["Female", "Male"],
  "native-country": [
    "United-States",
    "Cambodia",
    "England",
    "Puerto-Rico",
    "Canada",
    "Germany",
    "India",
    "Japan",
    "Greece",
    "China",
    "Cuba",
    "Iran",
    "Honduras",
    "Philippines",
    "Italy",
    "Poland",
    "Jamaica",
    "Vietnam",
    "Mexico",
    "Portugal",
    "Ireland",
    "France",
    "Dominican-Republic",
    "Laos",
    "Ecuador",
    "Taiwan",
    "Haiti",
    "Columbia",
    "Hungary",
    "Guatemala",
    "Nicaragua",
    "Scotland",
    "Thailand",
    "Yugoslavia",
    "El-Salvador",
    "Trinadad&Tobago",
    "Peru",
    "Hong",
    "Holand-Netherlands",
  ],
};


`income-ml-frontend/app/components/PredictForm.tsx`

In [ ]:
// ==============================
// IMPORTS
// ==============================

// Hook para manejar estado (state) dentro del componente
import { useState, useEffect } from "react";

// URL base del backend FastAPI
import { API_BASE } from "../lib/api";

// Tipos TypeScript que definen la estructura de datos
import type { PredictRequest, PredictResponse, FeaturesResponse } from "../lib/api";

// Opciones categóricas por defecto (si backend no responde)
import { DEFAULT_CATEGORICAL_OPTIONS } from "../lib/options";


// ==============================
// ESTADO INICIAL DEL FORMULARIO
// ==============================

// Objeto con valores iniciales del formulario
// Debe respetar exactamente la interfaz PredictRequest
const initialForm: PredictRequest = {
  age: 35,                    // Edad
  fnlwgt: 77516,              // Peso final del censo
  education_num: 10,          // Años de educación
  capital_gain: 0,            // Ganancia de capital
  capital_loss: 0,            // Pérdida de capital
  hours_per_week: 40,         // Horas trabajadas por semana
  workclass: "Private",       // Tipo de empleo
  education: "Bachelors",     // Nivel educativo
  marital_status: "Never-married",
  occupation: "Adm-clerical",
  relationship: "Not-in-family",
  race: "White",
  sex: "Male",
  native_country: "United-States",
};


// ==============================
// COMPONENTE PRINCIPAL
// ==============================

export function PredictForm() {

  // ------------------------------
  // STATES (ESTADOS DEL COMPONENTE)
  // ------------------------------

  // Estado del formulario (datos que el usuario edita)
  const [form, setForm] = useState<PredictRequest>(initialForm);

  // Opciones dinámicas para selects categóricos
  const [options, setOptions] =
    useState<Record<string, string[]>>(DEFAULT_CATEGORICAL_OPTIONS);

  // Resultado devuelto por el backend
  const [result, setResult] =
    useState<PredictResponse | null>(null);

  // Indica si se está esperando respuesta del backend
  const [loading, setLoading] = useState(false);

  // Guarda mensaje de error si ocurre algo
  const [error, setError] = useState<string | null>(null);

  // Estado de la API (sirve para mostrar advertencias)
  const [apiStatus, setApiStatus] =
    useState<"checking" | "ok" | "error">("checking");

  // Cargar opciones categóricas desde la API (si el modelo está entrenado)
  
  // ------------------------------
  // EFECTO: CARGAR OPCIONES DESDE API
  // ------------------------------

  // useEffect con [] significa:
  // Se ejecuta UNA sola vez cuando el componente se monta
  useEffect(() => {

    // Llamada HTTP al endpoint /features
    fetch(`${API_BASE}/features`)

      // Si la respuesta HTTP es 200 OK
      .then((res) => {
        if (res.ok) {
          return res.json() as Promise<FeaturesResponse>;
        }
        // Si no es OK, lanzamos error
        throw new Error("API no disponible");
      })

      // Si se pudo obtener JSON correctamente
      .then((data) => {

        // Si existen opciones categóricas entrenadas
        if (
          data.categorical_options &&
          Object.keys(data.categorical_options).length > 0
        ) {
          // Actualizamos opciones del estado
          setOptions(data.categorical_options);
        }

        // Marcamos API como funcional
        setApiStatus("ok");
      })

      // Si ocurre cualquier error (API apagada, etc.)
      .catch(() => {
        setApiStatus("error");
      });

  }, []); // array vacío = solo al montar


  // ------------------------------
  // MANEJO DE CAMBIOS EN INPUTS
  // ------------------------------

  const handleChange = (
    field: keyof PredictRequest,
    value: string | number
  ) => {

    // Actualiza únicamente el campo modificado
    setForm((prev) => ({
      ...prev,
      [field]: value,
    }));

    // Limpiar resultado y error anteriores
    setResult(null);
    setError(null);
  };


  // ------------------------------
  // ENVÍO DEL FORMULARIO
  // ------------------------------

  const handleSubmit = async (e: React.FormEvent) => {

    e.preventDefault(); // Evita que la página se recargue

    setLoading(true);   // Activa indicador de carga
    setError(null);     // Limpia errores previos
    setResult(null);    // Limpia resultado previo

    try {

      // Enviamos POST al backend
      const res = await fetch(`${API_BASE}/predict`, {
        method: "POST",
        headers: {
          "Content-Type": "application/json",
        },
        body: JSON.stringify(form), // Convertimos estado a JSON
      });

      // Convertimos respuesta a JSON
      const data = await res.json();

      // Si backend devuelve error HTTP
      if (!res.ok) {
        throw new Error(data.detail || "Error en la predicción");
      }

      // Guardamos resultado en estado
      setResult(data);

    } catch (err) {

      // Si ocurre error (red, backend, etc.)
      setError(
        err instanceof Error
          ? err.message
          : "Error de conexión"
      );

    } finally {

      // Se ejecuta siempre, haya error o no
      setLoading(false);
    }
  };

  // ------------------------------
  // LISTA DE VARIABLES CATEGÓRICAS
  // ------------------------------

  const categoricalKeys = [
    "workclass",
    "education",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native_country",
  ] as const;

  // Mapeo: nombre en el formulario -> clave en options (API usa "marital-status", form usa "marital_status")
  // ------------------------------
  // MAPEO DE NOMBRES
  // ------------------------------

  // Backend usa guiones (marital-status)
  // Frontend usa guion bajo (marital_status)
  
  const optionKey: Record<string, string> = {
    workclass: "workclass",
    education: "education",
    marital_status: "marital-status",
    occupation: "occupation",
    relationship: "relationship",
    race: "race",
    sex: "sex",
    native_country: "native-country",
  };

  // ==============================
  // RENDER (JSX)
  // ==============================


  return (
    <main className="min-h-screen bg-slate-50 dark:bg-slate-900 text-slate-800 dark:text-slate-200">
      <div className="max-w-2xl mx-auto px-4 py-8">
        <header className="mb-8 text-center">
          <h1 className="text-2xl font-bold text-slate-900 dark:text-white">
            Predicción de ingresos (Adult Dataset)
          </h1>
          <p className="text-slate-600 dark:text-slate-400 mt-1">
            Introduce los datos y el modelo predecirá si los ingresos son &gt;50K o ≤50K.
          </p>
          {apiStatus === "checking" && (
            <p className="text-amber-600 mt-2 text-sm">Comprobando conexión con la API…</p>
          )}
          {apiStatus === "error" && (
            <p className="text-amber-600 mt-2 text-sm">
              La API no está disponible. Asegúrate de ejecutar el backend: <code className="bg-slate-200 dark:bg-slate-700 px-1 rounded">python -m src.api</code>
            </p>
          )}
        </header>

        <form onSubmit={handleSubmit} className="space-y-6 bg-white dark:bg-slate-800 rounded-xl shadow-lg p-6">
          {/* Campos numéricos */}
          <section>
            <h2 className="text-lg font-semibold mb-3 text-slate-700 dark:text-slate-300">Datos numéricos</h2>
            <div className="grid grid-cols-1 sm:grid-cols-2 gap-4">
              <label className="block">
                <span className="block text-sm font-medium mb-1">Edad (age)</span>
                <input
                  type="number"
                  min={1}
                  max={120}
                  value={form.age}
                  onChange={(e) => handleChange("age", parseInt(e.target.value, 10) || 0)}
                  className="w-full rounded border border-slate-300 dark:border-slate-600 bg-white dark:bg-slate-700 px-3 py-2"
                />
              </label>
              <label className="block">
                <span className="block text-sm font-medium mb-1">Peso final (fnlwgt)</span>
                <input
                  type="number"
                  min={0}
                  value={form.fnlwgt}
                  onChange={(e) => handleChange("fnlwgt", parseInt(e.target.value, 10) || 0)}
                  className="w-full rounded border border-slate-300 dark:border-slate-600 bg-white dark:bg-slate-700 px-3 py-2"
                />
              </label>
              <label className="block">
                <span className="block text-sm font-medium mb-1">Años de educación (education-num)</span>
                <input
                  type="number"
                  min={1}
                  max={16}
                  value={form.education_num}
                  onChange={(e) => handleChange("education_num", parseInt(e.target.value, 10) || 0)}
                  className="w-full rounded border border-slate-300 dark:border-slate-600 bg-white dark:bg-slate-700 px-3 py-2"
                />
              </label>
              <label className="block">
                <span className="block text-sm font-medium mb-1">Ganancia de capital (capital-gain)</span>
                <input
                  type="number"
                  min={0}
                  value={form.capital_gain}
                  onChange={(e) => handleChange("capital_gain", parseInt(e.target.value, 10) || 0)}
                  className="w-full rounded border border-slate-300 dark:border-slate-600 bg-white dark:bg-slate-700 px-3 py-2"
                />
              </label>
              <label className="block">
                <span className="block text-sm font-medium mb-1">Pérdida de capital (capital-loss)</span>
                <input
                  type="number"
                  min={0}
                  value={form.capital_loss}
                  onChange={(e) => handleChange("capital_loss", parseInt(e.target.value, 10) || 0)}
                  className="w-full rounded border border-slate-300 dark:border-slate-600 bg-white dark:bg-slate-700 px-3 py-2"
                />
              </label>
              <label className="block">
                <span className="block text-sm font-medium mb-1">Horas por semana (hours-per-week)</span>
                <input
                  type="number"
                  min={1}
                  max={99}
                  value={form.hours_per_week}
                  onChange={(e) => handleChange("hours_per_week", parseInt(e.target.value, 10) || 0)}
                  className="w-full rounded border border-slate-300 dark:border-slate-600 bg-white dark:bg-slate-700 px-3 py-2"
                />
              </label>
            </div>
          </section>

          {/* Campos categóricos */}
          <section>
            <h2 className="text-lg font-semibold mb-3 text-slate-700 dark:text-slate-300">Datos categóricos</h2>
            <div className="grid grid-cols-1 sm:grid-cols-2 gap-4">
              {categoricalKeys.map((key) => {
                const opts = options[optionKey[key]] || [];
                const label = key.replace(/_/g, " ");
                return (
                  <label key={key} className="block">
                    <span className="block text-sm font-medium mb-1">{label}</span>
                    <select
                      value={form[key]}
                      onChange={(e) => handleChange(key, e.target.value)}
                      className="w-full rounded border border-slate-300 dark:border-slate-600 bg-white dark:bg-slate-700 px-3 py-2"
                    >
                      {opts.map((opt) => (
                        <option key={opt} value={opt}>{opt}</option>
                      ))}
                    </select>
                  </label>
                );
              })}
            </div>
          </section>

          <div className="flex flex-col sm:flex-row gap-3 pt-2">
            <button
              type="submit"
              disabled={loading}
              className="px-4 py-2 bg-blue-600 text-white rounded-lg font-medium hover:bg-blue-700 disabled:opacity-50 disabled:cursor-not-allowed"
            >
              {loading ? "Prediciendo…" : "Predecir ingresos"}
            </button>
            <button
              type="button"
              onClick={() => { setForm(initialForm); setResult(null); setError(null); }}
              className="px-4 py-2 border border-slate-300 dark:border-slate-600 rounded-lg font-medium hover:bg-slate-100 dark:hover:bg-slate-700"
            >
              Restablecer valores
            </button>
          </div>

          {error && (
            <div className="p-3 rounded-lg bg-red-100 dark:bg-red-900/30 text-red-800 dark:text-red-200 text-sm">
              {error}
            </div>
          )}

          {result && (
            <div className="p-4 rounded-lg bg-slate-100 dark:bg-slate-700 border border-slate-200 dark:border-slate-600">
              <h3 className="font-semibold mb-2">Resultado de la predicción</h3>
              <p className="text-lg">
                Ingresos previstos: <strong>{result.prediction}</strong>
              </p>
              <p className="text-sm text-slate-600 dark:text-slate-400 mt-1">
                Probabilidad de &gt;50K: {(result.probability * 100).toFixed(2)}%
              </p>
            </div>
          )}
        </form>
      </div>
    </main>
  );
}


`income-ml-frontend/app/routes/home.tsx`

In [ ]:
import type { Route } from "./+types/home";
import { PredictForm } from "../components/PredictForm";

export function meta({}: Route.MetaArgs) {
  return [
    { title: "Predicción de ingresos - Income ML" },
    { name: "description", content: "Aplicación de predicción de ingresos (>50K / ≤50K) con modelo de aprendizaje automático." },
  ];
}

export default function Home() {
  return <PredictForm />;
}


### Cómo ejecutarlo

En la raiz del proyecto

    `pip install -r requirements.txt`

    `python -m src.train`

    `python -m src.api`

Del lado del Front

    `cd income-ml-frontend`

    `npm install`

    `npm run dev`